# ಪಾಠ 01 - AI ಏಜೆಂಟ್ಗಳ ಪರಿಚಯ

**ಆರಂಭಿಕರಿಗಾಗಿ AI ಏಜೆಂಟ್ಗಳು** ಕೋರ್ಸ್‌ನ ಮೊದಲ ಪಾಠಕ್ಕೆ ಸ್ವಾಗತ!

**AI ಏಜೆಂಟ್** ಎಂದರೆ ಅದರ ತರ್ಕಯಂತ್ರವಾಗಿ ದೊಡ್ಡ ಭಾಷಾ ಮಾದರಿಯನ್ನು (LLM) ಬಳಸುವ ಮತ್ತು ವಾಸ್ತವ ಜಗತ್ತಿನಲ್ಲಿ *ಕೃತ್ಯಗಳನ್ನು* ಕೈಗೊಳ್ಳುವ — API ಗಳನ್ನು ಕರೆಸುವುದು, ಡೇಟಾಬೇಸ್‌ಗಳನ್ನು ವಿಚಾರಿಸುವುದು, ಅಥವಾ ಕೋಡ್ ಅನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುವುದು — ಬಳಕೆದಾರನ ಪರವಾಗಿ ಗುರಿಯನ್ನು ಸಾಧಿಸುವ ಕಾರ್ಯಕ್ರಮ.

ಈ ನೋಟ್ಬುಕ್ಸಿನಲ್ಲಿ ನೀವು ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸುವಿರಿ: **ಪ್ರವಾಸ ಏಜೆಂಟ್** ಮತ್ತು ಇದು ರಜೆ ತಾಣಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುತ್ತದೆ. ಅದರ ಮಾರ್ಗದಲ್ಲಿ ನೀವು ಹೇಗೆ ಎಂದು ಕಲಿಯುತ್ತೀರಿ:

1. **Microsoft Agent Framework** ಬಳಸಿ Azure AI Foundry Agent ಸೇವೆಯೊಂದಿಗೆ ಸಂಪರ್ಕ ಸಾಧಿಸುವುದು.
2. ಏಜೆಂಟ್‌ಗೆ ಒಂದು **ಪರಿಕರ** ನೀಡುವುದು — ಅದು ಕರೆಸಬಹುದಾದ ಸರಳ Python ಕಾರ್ಯ.
3. ಏಜೆಂಟ್ ಅನ್ನು ಕಾರ್ಯಗೊಳಿಸಿ ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸುವುದು.
4. ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಟೋಕೆನ್‌-ದ್ವಾರಾ ಹರಿಯಿಸುವುದು.


## ಸೆಟಪ್

ಈ ನೋಟುಬುಕ್ ಅನ್ನು 실행 ಮಾಡುವ ಮೊದಲು, ನೀವು ಖಚಿತಗೊಳಿಸಿಕೊಳ್ಳಿ:

1. **ನಿರ್ವಹಿಸಲಾದ ಚಾಟ್ ಮಾದರಿಯೊಂದಿಗೆ ಒಂದು Azure AI Foundry ಯೋಜನೆ** (ಉದಾ. `gpt-4o-mini`).
2. **Azure CLI ನಲ್ಲಿ ಲಾಗಿನ್ ಆಗಿರುವುದು** — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಅನ್ನು 실행 ಮಾಡಿ.
3. **ಅಗತ್ಯವಾದ ಪರಿಸರ ಚರಗಳನ್ನು ಹೊಂದಿಸಲಾಗಿದೆ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ Azure AI Foundry ಯೋಜನೆಯ ಅಂತ್ಯಬಿಂದುವು.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ನಿಯೋಜಿತ ಮಾದರಿಯ ಹೆಸರು.

ಕೆಳಗಿನ ಸೆಲ್ ನಿಮಗೆ ಬೇಕಾದ Python ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಸ್ಥಾಪಿಸುತ್ತದೆ.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ರಚನೆ

ಏಜೆಂಟ್‌ಗೆ ಎರಡು ವಿಷಯಗಳು ಅಗತ್ಯವಿದೆ:

- **ಸೂಚನೆಗಳು** ಅವನಿಗೆ *ಅವನು ಯಾರು* ಮತ್ತು *ಎಲ್ಲಿ ಹೇಗೆ ನಡೆದುಕೊಳ್ಳಬೇಕು* ಎಂದು ಹೇಳುವವು (ಒಂದು ಸಿಸ್ಟಮ್ ಪ್ರಾಂಪ್ಟ್).
- **ಉಪಕರಣಗಳು** — Python функцийಗಳನ್ನು `@tool` ಮೂಲಕ ಅಲಂಕೃತಗೊಳಿಸಲಾಗಿದ್ದು, ಏಜೆಂಟ್ ಮಾಹಿತಿ ಪಡೆಯಲು ಅಥವಾ ಕಾರ್ಯಗಳನ್ನು ಮಾಡಲು ಈ ಉಪಕರಣಗಳನ್ನು ಕರೆ ಮಾಡಬಹುದು.

ಈ ಕೆಳಜೊತೆ ನಾವು ಜನಪ್ರಿಯ ಪ್ರವಾಸ ಸ್ಥಳಗಳ ಪಟ್ಟಿಯನ್ನು ಹಿಂತಿರುಗಿಸುವ ಸರಳ ಉಪಕರಣವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತೇವೆ. ಬಳಕೆದಾರರು ಪ್ರವಾಸ ಸಲಹೆಗಳನ್ನು ಕೇಳಿದಾಗ ಏಜೆಂಟ್ ಈ ಉಪಕರಣವನ್ನು ಬಳಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ಸ್ಟ್ರೀಮಿಂಗ್ ಪ್ರತಿಕ್ರಿಯೆಗಳು

ಹೆಚ್ಚು интерактив್ ಅನುಭವಕ್ಕಾಗಿಯೆ ನೀವು ಏಜೆಂಟ್‍ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು **ಸ್ಟ್ರೀಮ್** ಮಾಡಬಹುದು. ಸಂಪೂರ್ಣ ಪ್ರತಿಕ್ರಿಯೆಗಾಗಿ ಕಾಯುವ ಬದಲು, ಏಜೆಂಟ್ ರಚಿಸಿರುವಂತೆ ಟೆಕ್ಸ್ಟ್ ತುಂಡುಗಳನ್ನು ನೀಡುತ್ತದೆ. ನೀವು ರಿಯಲ್ ಟೈಮ್‌ನಲ್ಲಿ output ಅನ್ನು ತೋರಿಸಲು ಬಯಸುವ ಚಾಟ್ ಇಂಟರ್‌ಫೇಸ್ಗಳಲ್ಲಿ ಇದು ವಿಶೇಷವಾಗಿ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಮಾಡಬೇಕೆಂದು ಕಲಿತಿರಿ:

- **ಪ್ರದಾತರನ್ನು ರಚಿಸು** Azure AI Foundry Agent Service ಗೆ ಸಂಪರ್ಕಿಸಲು `AzureAIProjectAgentProvider` ಬಳಸಿ.
- **ಒಂದು ಸಾಧನವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸು** `@tool` ಡೆಕೊರೇಟರ್‌ನೊಂದಿಗೆ, ಇದರಿಂದ ಏಜೆಂಟ್ ನಿಮ್ಮ Python ಕಾರ್ಯಗಳನ್ನು ಕರೆ ಮಾಡಬಹುದು.
- **ಏಜೆಂಟ್‌ನ್ನು ಚಾಲನೆಮಾಡು** ಬಳಕೆದಾರ ಸಂದೇಶದೊಂದಿಗೆ ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಮುದ್ರಿಸು.
- **ನೇವಿಗೆ ಉತ್ತರಗಳನ್ನು ಪ್ರವಾ ಹಿಸು** ನೈಜ ಕಾಲದ ಔಟ್‌ಪುಟ್‌ಗಾಗಿ.

ಮುಂದಿನ ಪಾಠದಲ್ಲಿ ನಾವು ಏಜೆಂಟಿಕ್ ರೂಪರೇಷೆಗಳನ್ನೂ ಹೆಚ್ಚು ಆಳವಾಗಿ ಪರಿಶೀಲಿಸುವೆವು ಮತ್ತು ಏಜೆಂಟ್‌ಗಳಿಗೆ ಹೆಚ್ಚು ಸಾಮರ್ಥ್ಯವಂತ ಸಾಧನಗಳನ್ನು ಹಾಗೂ ಬಹು ಹಂತ ಯುಕ್ತಿವಾದ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ನೀಡುವುದು ಹೇಗೆ ಎಂಬುದನ್ನು ಕಲಿಯುವೆವು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರಿಕೆ**:
ಈ ದಾಖಲೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಎಂಬ AI ಭಾಷಾಂತರ ಸೇವೆಯನ್ನು ಬಳಸಿಕೊಂಡು ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯುಂಟಾಗಲು ಪ್ರಯತ್ನಿಸಿದರೂ ಸಹ, ಸ್ವಯಂಚಾಲಿತ ಭಾಷಾಂತರಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳಿರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಮನಗಂಡಿರಬೆಕು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ದಸ್ತಾವೇಜನ್ನು ಅಧಿಕೃತ ಮತ್ತು ಪ್ರಾಮಾಣಿಕ ಮೂಲವಾಗಿ ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವನ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ. ಈ ಭಾಷಾಂತರವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಗ್ರಹಿಕೆಗಳು ಅಥವಾ ಅರ್ಥ ಕೊಳಪುಗಳಿಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
